# OpenAI API con Python

**Conceptos:** hacer llamadas a la API, usar instrucciones, JSON estructurado, y tools

## 1. Setup

Instalar dependencias.

Ejecutar una vez.

In [1]:
%pip install --upgrade openai pydantic

  Obtaining dependency information for openai from https://files.pythonhosted.org/packages/0a/bf/ccff9be562e24207716d04ef9dc931c76aff0c89a7265da43e2104d7fe06/openai-2.38.0-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 12.4 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: openai
    Found existing installation: openai 2.37.0
    Uninstalling openai-2.37.0:
      Successfully uninstalled openai-2.37.0

[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. API Key

Usar variable de entorno.

No hardcodear secrets.

In [2]:
import os
from getpass import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-nano-2026-03-17")

from openai import OpenAI

client = OpenAI()

print(f"Modelo: {OPENAI_MODEL}")
print("Cliente listo")

Modelo: gpt-5.4-nano-2026-03-17
Cliente listo


## 10. Más parámetros de configuración

En la `Responses API`, los parámetros de configuración se pasan en `client.responses.create(...)`.

Regla práctica:

- `temperature`: creatividad / variación.
- `top_p`: alternativa a `temperature`; normalmente se ajusta uno u otro.
- `max_output_tokens`: límite máximo de tokens generados.
- `text={"verbosity": ...}`: extensión o concisión de la respuesta.
- `reasoning={"effort": ...}`: esfuerzo de razonamiento en modelos compatibles.
- `stream=True`: recibir eventos mientras la respuesta se genera.
- `metadata`, `store`, `safety_identifier`: útiles para trazabilidad y producción.


In [3]:
# Prompt base para comparar configuraciones
prompt_config = """
Dame 5 ideas de mini proyectos para enseñar APIs a estudiantes de ingeniería.
Cada idea debe tener: nombre, objetivo y endpoint ejemplo.
"""

def imprimir_respuesta(titulo, response):
    print("=" * 80)
    print(titulo)
    print("=" * 80)
    print(response.output_text)
    print("\nUso de tokens:")
    print(response.usage)


### 10.1 `temperature`: más determinismo vs. más creatividad

Valores bajos suelen producir respuestas más enfocadas y repetibles. Valores altos generan más variedad.

> Recomendación: cambia `temperature` o `top_p`, pero no ambos al mismo tiempo.


In [4]:
for temp in [0.0, 0.7, 1.2]:
    response = client.responses.create(
        model=OPENAI_MODEL,
        input=prompt_config,
        temperature=temp,
        max_output_tokens=220,
    )

    imprimir_respuesta(f"temperature={temp}", response)


temperature=0.0
1) **Nombre:** Catálogo de Películas con Búsqueda  
   **Objetivo:** Enseñar consumo de APIs REST con parámetros (query), paginación básica y manejo de respuestas JSON.  
   **Endpoint ejemplo:** `GET https://www.omdbapi.com/?apikey=TU_API_KEY&s=matrix&page=1`

2) **Nombre:** Clima por Ubicación (Geocoding + Weather)  
   **Objetivo:** Practicar encadenamiento de APIs (convertir ciudad a coordenadas y luego consultar clima), manejo de errores y validación de inputs.  
   **Endpoint ejemplo:** `GET https://api.open-meteo.com/v1/forecast?latitude=-34.6037&longitude=-58.3816&current=temperature_2m`

3) **Nombre:** Seguimiento de Envíos (Tracking)  
   **Objetivo:** Enseñar autenticación (API key/token), consumo de endpoints con rutas dinámicas y modelado de estados (en tránsito, entregado, etc.).  
   **Endpoint ejemplo

Uso de tokens:
ResponseUsage(input_tokens=35, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=220, output_tokens_details=OutputTok

### 10.2 `max_output_tokens`: controlar longitud y costo

`max_output_tokens` pone un techo a la salida generada. Incluye tokens visibles y, en modelos de razonamiento, tokens internos de razonamiento.

Útil para:

- evitar respuestas demasiado largas;
- controlar costo;
- forzar resúmenes breves.


In [5]:
response = client.responses.create(
    model=OPENAI_MODEL,
    input="Explica en español qué es OAuth 2.0 para un principiante.",
    max_output_tokens=80,
)

imprimir_respuesta("Respuesta limitada con max_output_tokens=80", response)

# Si el modelo se corta por el límite, puede aparecer incomplete_details.
print("\nDetalles de incompletitud:", response.incomplete_details)


Respuesta limitada con max_output_tokens=80
OAuth 2.0 es un **método para que una aplicación acceda a recursos de otra** (por ejemplo, acceder a tu Google Drive) **sin que tengas que compartir tu contraseña**.

### ¿Por qué existe?
Imagina que una app de terceros quiere ver tu información en tu cuenta (como fotos o correo). En vez de pedirte tu contraseña, OAuth

Uso de tokens:
ResponseUsage(input_tokens=22, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=80, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=102)

Detalles de incompletitud: IncompleteDetails(reason='max_output_tokens')


### 10.3 `top_p`

`top_p` limita el conjunto de tokens candidatos por probabilidad acumulada.

- `top_p=1.0`: considera todo el espacio probable.
- valores menores: respuesta más restringida.

No lo combines normalmente con cambios de `temperature`.


In [ ]:
for top_p in [1.0, 0.5, 0.2]:
    response = client.responses.create(
        model=OPENAI_MODEL,
        input="Genera 6 nombres para una app que enseña programación con IA.",
        top_p=top_p,
        max_output_tokens=120,
    )

    imprimir_respuesta(f"top_p={top_p}", response)


### 10.4 `text.verbosity`: salida concisa, media o detallada

En modelos compatibles, `text={"verbosity": "low" | "medium" | "high"}` permite pedir respuestas más breves o más desarrolladas sin cambiar el prompt.


In [6]:
for verbosity in ["low", "medium", "high"]:
    try:
        response = client.responses.create(
            model=OPENAI_MODEL,
            input="Explica las diferencias entre REST, GraphQL y gRPC.",
            text={"verbosity": verbosity},
            max_output_tokens=350,
        )

        imprimir_respuesta(f"verbosity={verbosity}", response)

    except Exception as e:
        print(f"El modelo {OPENAI_MODEL} no aceptó text.verbosity={verbosity}:")
        print(type(e).__name__, e)


verbosity=low
Aquí van las diferencias principales entre **REST**, **GraphQL** y **gRPC**:

## REST
- **Modelo**: Arquitectura basada en recursos (URLs) y operaciones estándar.
- **Formato**: Normalmente **HTTP + JSON** (también puede usar otros).
- **Cómo consultas**: Se suelen hacer **múltiples requests** para obtener datos relacionados (por ejemplo, listar “users” y luego llamar “orders” por cada user).
- **Endpoint**: Muchas rutas fijas (por recurso/acción).
- **Ventajas**: Simple, muy conocido, fácil de probar con navegador/HTTP clients, buen soporte en la web.
- **Desventajas**: Puede haber **overfetching** (traer más datos de los necesarios) o **underfetching** (necesitar más requests).

## GraphQL
- **Modelo**: Es un **lenguaje/endpoint único** con un **esquema (schema)** que define los tipos y relaciones.
- **Formato**: Típicamente **HTTP + JSON** (también soporta WebSockets para subscripciones).
- **Cómo consultas**: El cliente pide exactamente los campos que necesita en una 

### 10.5 `reasoning.effort`: esfuerzo de razonamiento

En modelos compatibles con razonamiento, `reasoning={"effort": "low" | "medium" | "high"}` permite balancear latencia/costo vs. profundidad.

Usa:

- `low`: preguntas simples, menor latencia.
- `medium`: análisis general.
- `high`: problemas complejos, planificación, debugging difícil.


In [7]:
problema = """
Tengo una API que recibe 10.000 requests/minuto.
A veces devuelve 500, pero solo bajo carga.
Dame un plan de diagnóstico ordenado por prioridad.
"""

for effort in ["low", "medium", "high"]:
    try:
        response = client.responses.create(
            model=OPENAI_MODEL,
            input=problema,
            reasoning={"effort": effort},
            max_output_tokens=350,
        )

        imprimir_respuesta(f"reasoning.effort={effort}", response)

    except Exception as e:
        print(f"El modelo {OPENAI_MODEL} no aceptó reasoning.effort={effort}:")
        print(type(e).__name__, e)


reasoning.effort=low
Aquí tienes un **plan de diagnóstico ordenado por prioridad** para un problema de **500 solo bajo carga** (10.000 requests/min). La idea es **reducir el “tiempo a causa raíz”**: primero asegurar observabilidad y luego aislar qué componente falla (app / red / LB / DB / colas / dependencias).

---

## 1) Confirmar y cuantificar el fallo (antes de tocar código)
**Objetivo:** tener certeza de *cuándo* y *cuánto* falla y bajo qué patrón.

- Verifica métricas y umbrales:
  - % de 500 por minuto (y por endpoint / método).
  - Latencia (p50/p95/p99) en paralelo.
  - Saturación de recursos: CPU, RAM, GC, threads, heap, file descriptors.
  - Errores “upstream”: timeouts, resets, 502/504 en el LB/CDN si existen.
- Identifica “carga” real: tasa objetivo vs tasa efectiva (por ejemplo, picos, reintentos del cliente, burst).

**Resultado esperado:** una tabla tipo “endpoint X tiene 500 cuando p95 > Y y CPU > Z” o “cuando hay picos de concurrencia”.

---

## 2) Asegurar observabil

### 10.6 Combinar parámetros para un caso real

Ejemplo: generar una respuesta de soporte técnica, controlando tono, longitud, creatividad y trazabilidad.


In [8]:
ticket = """
Cliente: No puedo iniciar sesión. Me dice token inválido.
Ya borré caché y probé otro navegador.
Necesito entrar para descargar facturas.
"""

response = client.responses.create(
    model=OPENAI_MODEL,
    instructions="""
    Sos agente de soporte técnico.
    Respondé en español rioplatense, amable, claro y accionable.
    No prometas soluciones que no puedas verificar.
    """,
    input=ticket,
    temperature=0.3,
    max_output_tokens=250,
    text={"verbosity": "medium"},
    metadata={
        "curso": "openai-api",
        "modulo": "parametros",
        "tipo": "soporte"
    },
    store=False,
)

imprimir_respuesta("Respuesta de soporte configurada", response)


Respuesta de soporte configurada
Entiendo. Si ya probaste borrar caché y otro navegador y sigue apareciendo **“token inválido”**, lo más común es que el problema sea del **token de sesión** (caducado/corrupto) o de **cookies/almacenamiento** que no se están eliminando del todo.

Probemos en este orden (rápido y accionable):

### 1) Asegurá que se borren cookies del sitio (no solo “caché”)
1. Abrí el navegador en modo normal.
2. Configuración → Privacidad/Seguridad → **Cookies y otros datos del sitio**.
3. Buscá y eliminá los datos de **ese sitio** (la web donde iniciás sesión).
4. Cerrá y volvé a abrir el navegador.
5. Probá iniciar sesión de nuevo.

> Si usás Chrome/Edge: “Borrar datos de navegación” → marcá **Cookies y otros datos** (además de caché).

### 2) Desactivá temporalmente extensiones que tocan sesión
Especialmente: bloqueadores de trackers, adblock, antivirus web, extensiones de privacidad.
- Probá en **

Uso de tokens:
ResponseUsage(input_tokens=79, input_tokens_details=I

### 10.7 Salida JSON con `text.format`

Además de `client.responses.parse(...)`, también podés pedir JSON Schema directamente con `text={"format": ...}`.

Esto sirve cuando querés que la respuesta cumpla una estructura exacta.


In [9]:
schema_proyecto = {
    "type": "object",
    "properties": {
        "titulo": {"type": "string"},
        "nivel": {"type": "string", "enum": ["inicial", "intermedio", "avanzado"]},
        "objetivo": {"type": "string"},
        "endpoints": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "metodo": {"type": "string", "enum": ["GET", "POST", "PUT", "PATCH", "DELETE"]},
                    "path": {"type": "string"},
                    "descripcion": {"type": "string"}
                },
                "required": ["metodo", "path", "descripcion"],
                "additionalProperties": False
            }
        }
    },
    "required": ["titulo", "nivel", "objetivo", "endpoints"],
    "additionalProperties": False
}

response = client.responses.create(
    model=OPENAI_MODEL,
    input="Diseña un mini proyecto de API para gestionar reservas de laboratorio.",
    text={
        "format": {
            "type": "json_schema",
            "name": "mini_proyecto_api",
            "schema": schema_proyecto,
            "strict": True,
        }
    },
    temperature=0.2,
    max_output_tokens=500,
)

print(response.output_text)


{"titulo":"Mini proyecto de API: Gestión de Reservas de Laboratorio","nivel":"intermedio","objetivo":"Desarrollar una API REST para crear, consultar, actualizar y cancelar reservas de laboratorio, gestionando disponibilidad por laboratorio, fecha y franja horaria, y permitiendo el control básico de estado (pendiente/confirmada/cancelada).","endpoints":[{"metodo":"GET","path":"/health","descripcion":"Verifica el estado del servicio (health check)."},{"metodo":"POST","path":"/auth/login","descripcion":"Autentica un usuario y devuelve un token (si aplica en el proyecto)."},{"metodo":"GET","path":"/laboratorios","descripcion":"Lista laboratorios disponibles (id, nombre, ubicación, capacidad, etc.)."},{"metodo":"POST","path":"/laboratorios","descripcion":"Crea un laboratorio (solo roles autorizados)."},{"metodo":"GET","path":"/laboratorios/{laboratorioId}","descripcion":"Obtiene el detalle de un laboratorio."},{"metodo":"GET","path":"/reservas","descripcion":"Lista reservas con filtros: lab

### 10.8 Streaming con `stream=True`

Streaming sirve para empezar a mostrar texto antes de que termine toda la generación.

Es ideal para interfaces tipo chat.


In [10]:
stream = client.responses.create(
    model=OPENAI_MODEL,
    input="Escribe una explicación breve de qué es streaming en APIs, con una analogía.",
    stream=True,
    max_output_tokens=180,
)

for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)


El **streaming en APIs** es un modo de enviar y recibir datos **de manera continua**, en lugar de esperar a que el servidor termine y entregue todo el resultado de una vez. Así, el cliente puede empezar a procesar los datos **tan pronto como van llegando**.

**Analogía:** imagina que estás en una cafetería en vez de pedir “un paquete” con el café ya hecho. Con streaming, es como **chorros de café que se sirven poco a poco**: en cuanto cae la primera parte, ya puedes empezar a tomarla, y el resto sigue llegando en tiempo real.

### 10.9 `tool_choice` y `max_tool_calls`

Cuando usás tools, podés controlar si el modelo puede decidir libremente, si debe llamar una herramienta o si no puede llamarla.

`max_tool_calls` limita cuántas llamadas a herramientas integradas puede hacer en una respuesta.


In [11]:
# Ejemplo con web_search_preview.
# Usar tool_choice="required" fuerza a que el modelo use alguna herramienta disponible.

response = client.responses.create(
    model=OPENAI_MODEL,
    input="Busca una actualización reciente sobre modelos de IA y resúmela en 3 bullets.",
    tools=[{"type": "web_search_preview"}],
    tool_choice="required",
    max_tool_calls=1,
    max_output_tokens=250,
)

imprimir_respuesta("Web search forzado con max_tool_calls=1", response)


Web search forzado con max_tool_calls=1
- **OpenAI:** ChatGPT pasó a **GPT‑5.5 Instant** como modelo por defecto (y hay más disponibilidad/actualizaciones vinculadas a esa línea de modelos). ([help.openai.com](https://help.openai.com/en/articles/6825453-release-notes?utm_source=openai))  
- **Google / DeepMind:** En el marco de **Google I/O 2026**, se reportaron avances en **Gemini** (incluida la familia **Gemini 3.5**) y cambios de producto como la adopción de **“3.5 Flash”** en el **app de Gemini / Search**. ([blog.google](https://blog.google/intl/en-africa/products/explore-get-answers/gemini-3-5/?utm_source=openai)

Uso de tokens:
ResponseUsage(input_tokens=8495, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=250, output_tokens_details=OutputTokensDetails(reasoning_tokens=94), total_tokens=8745)


### 10.10 Configuración reusable

Podés centralizar configuraciones para no repetir parámetros en cada llamada.


In [12]:
CONFIG_BREVE = {
    "temperature": 0.2,
    "max_output_tokens": 180,
    "text": {"verbosity": "low"},
    "store": False,
}

CONFIG_CREATIVA = {
    "temperature": 1.0,
    "max_output_tokens": 350,
    "text": {"verbosity": "high"},
    "store": False,
}

def responder(prompt, config=None):
    config = config or {}
    response = client.responses.create(
        model=OPENAI_MODEL,
        input=prompt,
        **config,
    )
    return response.output_text

print("--- Breve ---")
print(responder("Dame una analogía para explicar embeddings.", CONFIG_BREVE))

print("\n--- Creativa ---")
print(responder("Dame una analogía para explicar embeddings.", CONFIG_CREATIVA))


--- Breve ---
Imagina que cada **embedding** es como una **etiqueta de dirección** para una palabra o imagen: no dice “qué es” en texto, sino que la convierte en un **punto en un mapa**.

- Dos cosas con **significado parecido** (p. ej., “perro” y “gato”) quedan **cerca** en ese mapa.  
- Cosas con **significado distinto** quedan **lejos**.  

Así, cuando el modelo busca algo “parecido”, en realidad está buscando **puntos cercanos** en ese espacio de embeddings.

--- Creativa ---
Piensa en los **embeddings** como una **tarjeta de identificación (tipo pasaporte) para palabras o imágenes**, pero en vez de usar solo texto, los convierten en un **vector numérico** que captura “cómo se parecen” entre sí.

### Analogía: “Mapa de gustos”
Imagina que vives en un mundo donde existen dos cosas:

- Un **mapa enorme** con millones de lugares.
- Cada palabra (o cada imagen) tiene su propio “lugar” en ese mapa.

Los **embeddings** son la forma de decir:
- “Si esta palabra fuera una ubicación, ¿a qué

## 11. Ejercicios de prompt engineering

Técnicas que vamos a practicar:

- **Zero-shot**: pedir una tarea sin ejemplos.
- **Few-shot**: incluir ejemplos de entrada/salida para enseñar el patrón.
- **Delimitadores**: separar instrucciones, contexto y datos con Markdown/XML.
- **Rol + reglas**: usar `instructions` para fijar comportamiento.



### 11.1 Helper para probar prompts

Vamos a crear una función pequeña para no repetir código.


In [13]:
def probar_prompt(titulo, prompt, instructions=None, temperature=0.2, max_output_tokens=300):
    print("=" * 90)
    print(titulo)
    print("=" * 90)

    kwargs = {
        "model": OPENAI_MODEL,
        "input": prompt,
        "temperature": temperature,
        "max_output_tokens": max_output_tokens,
    }

    if instructions:
        kwargs["instructions"] = instructions

    response = client.responses.create(**kwargs)
    print(response.output_text)
    return response


### 11.2 Ejercicio zero-shot: clasificación sin ejemplos

**Consigna:** clasificá un ticket de soporte usando solo instrucciones.

Probá cambiar el prompt para que la salida sea más consistente.


In [14]:
ticket = """
La app me cobra dos veces la suscripción mensual.
Ya revisé el resumen de la tarjeta y aparecen dos cargos del mismo día.
Necesito que lo solucionen hoy.
"""

prompt_zero_shot = f"""
Clasifica el siguiente ticket de soporte.

Categorías posibles: login, pagos, bug, consulta, otro.
Prioridades posibles: baja, media, alta.

Devuelve:
- categoría
- prioridad
- resumen en una frase

Ticket:
{ticket}
"""

probar_prompt(
    "Zero-shot: clasificación de ticket",
    prompt_zero_shot,
    temperature=0.0,
    max_output_tokens=180,
)


Zero-shot: clasificación de ticket
- **categoría:** pagos  
- **prioridad:** alta  
- **resumen en una frase:** La app está cobrando dos veces la suscripción mensual en la misma fecha y el usuario solicita una solución hoy.


Response(id='resp_08df7ed6c2134f64006a10a8be68d88196a3385e38fdc34b6c', created_at=1779476670.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-nano-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_08df7ed6c2134f64006a10a8bed9348196bb4571178eb805f5', content=[ResponseOutputText(annotations=[], text='- **categoría:** pagos  \n- **prioridad:** alta  \n- **resumen en una frase:** La app está cobrando dos veces la suscripción mensual en la misma fecha y el usuario solicita una solución hoy.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=0.0, tool_choice='auto', tools=[], top_p=0.98, background=False, completed_at=1779476671.0, conversation=None, max_output_tokens=180, max_tool_calls=None, previous_response_id=None, prompt=None, prompt_cache_key=None, prompt_cache_retention='in_memory', reasoning=Reasoning(effort='none', generate_

### 11.3 Ejercicio few-shot: enseñar el formato con ejemplos

**Consigna:** compará esta salida con la anterior.

Few-shot suele ser útil cuando querés que el modelo imite un criterio, tono o formato específico.


In [15]:
prompt_few_shot = f"""
Clasifica tickets de soporte en una sola línea con este formato exacto:
Categoria=<categoria>; Prioridad=<prioridad>; Resumen=<resumen breve>

Ejemplos:

Ticket: No puedo iniciar sesión. Dice contraseña incorrecta aunque la cambié.
Salida: Categoria=login; Prioridad=media; Resumen=Usuario no puede acceder tras cambio de contraseña

Ticket: La factura vino con un cargo duplicado y necesito corregirlo urgente.
Salida: Categoria=pagos; Prioridad=alta; Resumen=Posible cobro duplicado que requiere revisión urgente

Ticket: ¿Dónde puedo descargar el manual de uso?
Salida: Categoria=consulta; Prioridad=baja; Resumen=Usuario solicita ubicación del manual

Ahora clasifica este ticket:
Ticket: {ticket}
Salida:
"""

probar_prompt(
    "Few-shot: clasificación con ejemplos",
    prompt_few_shot,
    temperature=0.0,
    max_output_tokens=120,
)


Few-shot: clasificación con ejemplos
Categoria=pagos; Prioridad=alta; Resumen=La app cobra dos veces la suscripción mensual con dos cargos del mismo día, requiere solución hoy


Response(id='resp_0e79315fca33a246006a10a8c5b37c81a3ba8535b9c6c1e978', created_at=1779476677.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-nano-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_0e79315fca33a246006a10a8c5e55081a395ada3ee3a43141d', content=[ResponseOutputText(annotations=[], text='Categoria=pagos; Prioridad=alta; Resumen=La app cobra dos veces la suscripción mensual con dos cargos del mismo día, requiere solución hoy', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=0.0, tool_choice='auto', tools=[], top_p=0.98, background=False, completed_at=1779476678.0, conversation=None, max_output_tokens=120, max_tool_calls=None, previous_response_id=None, prompt=None, prompt_cache_key=None, prompt_cache_retention='in_memory', reasoning=Reasoning(effort='none', generate_summary=None, summary=None), safety_identifier=None,

### 11.4 Ejercicio con rol, reglas y tono

**Consigna:** redactá una respuesta al cliente.

Primero probá el prompt tal como está. Después agregá reglas como:

- no culpar al usuario;
- pedir solo la información mínima necesaria;
- no prometer reembolsos sin validación;
- máximo 120 palabras.


In [ ]:
instructions_soporte = """
Sos un agente senior de soporte SaaS.
Tu tono es amable, claro y profesional.
No prometas acciones que dependan de otro equipo.
No inventes datos que no estén en el ticket.
"""

prompt_respuesta_cliente = f"""
Redacta una respuesta para este cliente.

Ticket del cliente:
{ticket}
"""

probar_prompt(
    "Rol + reglas: respuesta de soporte",
    prompt_respuesta_cliente,
    instructions=instructions_soporte,
    temperature=0.3,
    max_output_tokens=220,
)


### 11.5 Ejercicio con delimitadores Markdown/XML

**Consigna:** el modelo debe usar solo la información del contexto.

Los delimitadores ayudan a separar instrucciones de datos. Esto es especialmente útil cuando pegás textos largos, documentos o contenido de usuarios.


In [ ]:
contexto_producto = """
Plan Basic:
- Hasta 3 usuarios.
- Soporte por email.
- 5 GB de almacenamiento.

Plan Pro:
- Hasta 25 usuarios.
- Soporte por chat y email.
- 100 GB de almacenamiento.
- Integraciones con Slack y Google Drive.

Plan Enterprise:
- Usuarios ilimitados.
- SSO/SAML.
- Soporte dedicado.
- Almacenamiento personalizado.
"""

pregunta_usuario = "¿Qué plan me conviene si somos 12 personas y necesitamos Slack?"

prompt_delimitado = f"""
# Instrucciones
Responde la pregunta usando únicamente el contexto provisto.
Si falta información, dilo explícitamente.
Devuelve una recomendación y una breve justificación.

<contexto>
{contexto_producto}
</contexto>

<pregunta>
{pregunta_usuario}
</pregunta>
"""

probar_prompt(
    "Delimitadores: usar solo contexto",
    prompt_delimitado,
    temperature=0.0,
    max_output_tokens=180,
)


### 11.6 Ejercicio: prompt con contexto + tarea + formato

**Consigna:** armá un prompt completo usando esta plantilla:

```text
# Rol
...

# Tarea
...

# Contexto
...

# Restricciones
...

# Formato de salida
...
```

Después ejecutalo y ajustá una sección por vez.


In [ ]:
prompt_plantilla = """
# Rol
Sos un tutor de Python para estudiantes de ingeniería.

# Tarea
Explicar list comprehensions y proponer ejercicios.

# Contexto
El estudiante ya conoce listas, loops for y condicionales if.
Todavía no usó sintaxis avanzada de Python.

# Restricciones
No uses jerga innecesaria.
Incluye ejemplos ejecutables.
No uses más de 500 palabras.

# Formato de salida
1. Explicación breve
2. Ejemplo básico
3. Ejemplo con condición
4. Tres ejercicios
5. Soluciones esperadas
"""

probar_prompt(
    "Plantilla completa de prompt",
    prompt_plantilla,
    temperature=0.2,
    max_output_tokens=700,
)
